# FraudeCero · Grupo 6 · 02 · Análisis (Spark SQL) y capa ORO
**Proyecto Integrador Big Data – Uniciencia 2026-II** · Integrantes: Juan David Montoya · William Conde Villamizar


**Objetivo:** responder las 3 preguntas de negocio con Spark SQL sobre la capa plata y publicar las tablas **oro** que alimentan el tablero.

| Pregunta de negocio | KPI asociado | Fórmula · unidad · fuente |
|---|---|---|
| **P1. ¿Cuánto** dinero y qué proporción de las transacciones se pierde por fraude? | **KPI1 – Tasa de fraude** | fraudes / transacciones × 100 · % · plata.transacciones |
| **P2. ¿Dónde** (en qué horas, rangos de monto y entidades colombianas) se concentra el riesgo? | **KPI2 – Monto fraudado** | Σ monto de transacciones con Class = 1 (y % del monto total) · EUR · plata.transacciones |
| **P3. ¿Por qué** una transacción resulta fraudulenta y qué tan bien se puede detectar? | **KPI3 – Fraudes detectados (recall) con precisión mínima** | VP / (VP + FN) × 100 · % · oro.predicciones (notebook 04a) |

Consultas: C1 (P1), C2 con **función de ventana** (P2), C3 (P2), C4 con **join entre las 2 fuentes** y ventana (P2), C5 con ventana LAG (P1, contexto Colombia), C6 (P3). Cada consulta termina con 2 renglones de interpretación generados con sus propios resultados.

## Configuración común
Esta celda es igual en todos los notebooks. Detecta si se ejecuta en **Google Colab** o en **Jupyter / GitHub Actions**, instala PySpark 3.5 + Delta Lake (OSS) y define:
- `LAKE`: carpeta del Lakehouse (capas `bronce`, `plata`, `oro`).
- `EVID`: carpeta `99_evidencias` (tablas de evidencia, log de monitoreo, tableros HTML).
- `guardar()` / `leer()`: escriben y leen tablas **Delta** (si Delta no está disponible usan Parquet y lo avisan).
- `registrar()`: escribe en el log de monitoreo el tiempo, las filas, el % de nulos y la memoria de cada etapa.

> En Colab cada notebook corre en una máquina distinta, por eso el Lakehouse se respalda en Google Drive al final de cada notebook y se restaura al inicio del siguiente.

In [ ]:
# ===================== CONFIGURACIÓN COMÚN FRAUDECERO =====================
NOTEBOOK = "02_analisis"
import os, sys, time, json, shutil, datetime, subprocess, hashlib
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub", "psutil", "plotly"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = "/content/drive/MyDrive/FraudeCero"
    LAKE = "/content/lakehouse"                      # disco local (rápido)
    LAKE_RESPALDO = f"{BASE_DRIVE}/lakehouse"        # respaldo persistente en Drive
    EVID = f"{BASE_DRIVE}/99_evidencias"
else:
    RAIZ = os.environ.get("FRAUDECERO_RAIZ", os.path.abspath(".."))
    LAKE = os.environ.get("FRAUDECERO_LAKE", os.path.join(RAIZ, "lakehouse"))
    LAKE_RESPALDO = None
    EVID = os.environ.get("FRAUDECERO_EVID", os.path.join(RAIZ, "99_evidencias"))

for carpeta in [LAKE, EVID, f"{EVID}/monitoreo", f"{EVID}/tablas"]:
    os.makedirs(carpeta, exist_ok=True)

def restaurar_lake():
    """En Colab: trae el Lakehouse desde Drive si el disco local está vacío."""
    if LAKE_RESPALDO and os.path.exists(LAKE_RESPALDO) and not os.listdir(LAKE):
        shutil.copytree(LAKE_RESPALDO, LAKE, dirs_exist_ok=True)
        print("Lakehouse restaurado desde Drive")

def respaldar_lake():
    """En Colab: copia el Lakehouse a Drive (sin la zona landing de archivos crudos)."""
    if LAKE_RESPALDO:
        shutil.copytree(LAKE, LAKE_RESPALDO, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("landing"))
        print("Lakehouse respaldado en Drive:", LAKE_RESPALDO)

restaurar_lake()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

FORMATO = "delta"
try:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
               .config("spark.driver.memory", "6g")
               .config("spark.sql.shuffle.partitions", "8")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.range(1).write.format("delta").mode("overwrite").save(f"{LAKE}/_prueba_delta")
except Exception as e:
    print("⚠️ Delta Lake no disponible, se usará Parquet. Detalle:", str(e)[:300])
    FORMATO = "parquet"
    spark = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
             .config("spark.driver.memory", "6g")
             .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ruta(capa, tabla):
    return f"{LAKE}/{capa}/{tabla}"

def leer(capa, tabla):
    df = spark.read.format(FORMATO).load(ruta(capa, tabla))
    df.createOrReplaceTempView(f"{capa}_{tabla}")
    return df

def guardar(df, capa, tabla, particion=None):
    w = df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.save(ruta(capa, tabla))
    return leer(capa, tabla)

# ---------- Monitoreo: tiempo, filas, % nulos y memoria por etapa ----------
import psutil, pandas as pd
LOG_MON = f"{EVID}/monitoreo/log_ejecuciones.csv"
ARCHIVO_RUN = f"{EVID}/monitoreo/run_actual.txt"
RUN_ID = os.environ.get("FRAUDECERO_RUN_ID")
if not RUN_ID:
    if NOTEBOOK == "00_ingesta" or not os.path.exists(ARCHIVO_RUN):
        RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # cada corrida del pipeline empieza en 00
        with open(ARCHIVO_RUN, "w") as f:
            f.write(RUN_ID)
    else:
        RUN_ID = open(ARCHIVO_RUN).read().strip()
ORIGEN = "github_actions" if os.environ.get("GITHUB_ACTIONS") else ("colab" if EN_COLAB else "jupyter")

def memoria_mb():
    py = psutil.Process(os.getpid()).memory_info().rss / 1024**2
    try:
        rt = spark.sparkContext._jvm.java.lang.Runtime.getRuntime()
        jvm = (rt.totalMemory() - rt.freeMemory()) / 1024**2
    except Exception:
        jvm = 0.0
    return round(py + jvm, 1)

def registrar(etapa, t0, df=None, filas=None):
    """Registra una etapa en el log de monitoreo. t0 = time.time() al inicio de la etapa."""
    duracion = round(time.time() - t0, 2)
    pct_nulos = None
    if df is not None:
        filas = df.count()
        cols = df.columns
        nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).collect()[0]
        pct_nulos = round(sum(v or 0 for v in nulos) / (filas * len(cols)) * 100, 4) if filas else 0.0
    fila = {"run_id": RUN_ID, "origen": ORIGEN, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "notebook": NOTEBOOK, "etapa": etapa, "duracion_s": duracion, "filas": filas,
            "pct_nulos": pct_nulos, "memoria_mb": memoria_mb(), "cpu_pct": psutil.cpu_percent(interval=0.2),
            "formato": FORMATO}
    pd.DataFrame([fila]).to_csv(LOG_MON, mode="a", index=False, header=not os.path.exists(LOG_MON))
    print(f"⏱️ {etapa}: {duracion} s · filas={filas} · nulos={pct_nulos}% · mem={fila['memoria_mb']} MB")

def mostrar(df, n=20):
    """display() de Databricks no existe en Jupyter/Colab: se muestra como tabla pandas."""
    from IPython.display import display
    display(df.limit(n).toPandas() if hasattr(df, "toPandas") else df)

print(f"Notebook: {NOTEBOOK} · Entorno: {ORIGEN} · Formato de tablas: {FORMATO.upper()} · Spark {spark.version}")
print("LAKE =", LAKE, "| EVID =", EVID)

In [ ]:
plata_cc = leer("plata", "transacciones")      # vista: plata_transacciones
plata_sfc = leer("plata", "sfc_tarjetas")      # vista: plata_sfc_tarjetas
plata_exp = leer("plata", "exposicion")        # vista: plata_exposicion
sql = lambda q: spark.sql(q)

### C1 · ¿Cuánto se pierde por fraude? (P1 · KPI1 y KPI2)

In [ ]:
t0 = time.time()
c1 = sql("""
SELECT COUNT(*)                                             AS transacciones,
       SUM(Class)                                           AS fraudes,
       ROUND(SUM(Class) / COUNT(*) * 100, 4)                AS kpi1_tasa_fraude_pct,
       ROUND(SUM(Amount), 2)                                AS monto_total_eur,
       ROUND(SUM(CASE WHEN Class = 1 THEN Amount END), 2)   AS kpi2_monto_fraudado_eur,
       ROUND(SUM(CASE WHEN Class = 1 THEN Amount END) / SUM(Amount) * 100, 4) AS pct_monto_fraudado,
       ROUND(AVG(CASE WHEN Class = 1 THEN Amount END), 2)   AS monto_medio_fraude_eur,
       ROUND(AVG(CASE WHEN Class = 0 THEN Amount END), 2)   AS monto_medio_normal_eur
FROM plata_transacciones
""")
k = c1.toPandas().iloc[0]
mostrar(c1)
guardar(c1, "oro", "kpis")
registrar("C1_kpis", t0)
print(f"📝 De {k.transacciones:,.0f} transacciones, {k.fraudes:,.0f} son fraude: una tasa de {k.kpi1_tasa_fraude_pct:.3f} %, es decir, 1 de cada {1/(k.kpi1_tasa_fraude_pct/100):,.0f}.")
print(f"📝 El fraude suma {k.kpi2_monto_fraudado_eur:,.2f} EUR ({k.pct_monto_fraudado:.3f} % del monto total) en solo 2 días; el fraude medio ({k.monto_medio_fraude_eur:.2f} EUR) supera al pago normal medio ({k.monto_medio_normal_eur:.2f} EUR).")

### C2 · ¿A qué horas se concentra el fraude? (P2 · función de ventana)
`RANK()` ordena las horas por tasa de fraude y `SUM() OVER` acumula la participación de cada hora en el total de fraudes.

In [ ]:
t0 = time.time()
c2 = sql("""
WITH por_hora AS (
  SELECT hora, COUNT(*) AS transacciones, SUM(Class) AS fraudes,
         ROUND(SUM(Class) / COUNT(*) * 100, 4) AS tasa_fraude_pct,
         ROUND(SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END), 2) AS monto_fraudado_eur
  FROM plata_transacciones GROUP BY hora)
SELECT hora, transacciones, fraudes, tasa_fraude_pct, monto_fraudado_eur,
       RANK() OVER (ORDER BY tasa_fraude_pct DESC) AS ranking_riesgo,
       ROUND(SUM(fraudes) OVER (ORDER BY fraudes DESC, hora ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
             / SUM(fraudes) OVER () * 100, 2) AS pct_acumulado_fraudes
FROM por_hora ORDER BY hora
""")
guardar(c2, "oro", "fraude_por_hora")
h = c2.toPandas()
mostrar(c2.orderBy("ranking_riesgo"), 24)
registrar("C2_fraude_hora", t0)
top = h.sort_values("ranking_riesgo").head(3)
madrug = h[h.hora.between(0, 5)]
print(f"📝 Las horas de mayor riesgo son {', '.join(str(x)+':00' for x in top.hora)} con tasas de hasta {top.tasa_fraude_pct.max():.3f} %, frente a una tasa global de {k.kpi1_tasa_fraude_pct:.3f} %.")
print(f"📝 La madrugada (0–5 h) tiene solo {madrug.transacciones.sum()/h.transacciones.sum()*100:.1f} % de las transacciones pero {madrug.fraudes.sum()/h.fraudes.sum()*100:.1f} % de los fraudes: conviene reforzar controles en esa franja.")

### C3 · ¿En qué rangos de monto se concentra el fraude? (P2)

In [ ]:
t0 = time.time()
c3 = sql("""
SELECT rango_monto, COUNT(*) AS transacciones, SUM(Class) AS fraudes,
       ROUND(SUM(Class) / COUNT(*) * 100, 4) AS tasa_fraude_pct,
       ROUND(SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END), 2) AS monto_fraudado_eur,
       ROUND(SUM(Class) / SUM(SUM(Class)) OVER () * 100, 2) AS pct_de_los_fraudes
FROM plata_transacciones GROUP BY rango_monto ORDER BY rango_monto
""")
guardar(c3, "oro", "fraude_por_monto")
m = c3.toPandas()
mostrar(c3)
registrar("C3_fraude_monto", t0)
peor = m.sort_values("tasa_fraude_pct", ascending=False).iloc[0]
mas = m.sort_values("fraudes", ascending=False).iloc[0]
print(f"📝 El rango con mayor tasa de fraude es {peor.rango_monto[3:]} EUR ({peor.tasa_fraude_pct:.3f} %), y el que concentra más casos es {mas.rango_monto[3:]} EUR ({mas.pct_de_los_fraudes:.1f} % de los fraudes).")
print("📝 El monto por sí solo no separa el fraude: hay fraudes en todos los rangos, por eso se necesita el modelo con las variables V.")

### C4 · ¿Dónde está la exposición en Colombia? (P2 · **join entre 2 fuentes** + ventana)
Une la SFC (transacciones de compra con tarjeta de crédito por entidad) con la tasa de fraude de F1 y rankea las entidades del último año completo.

In [ ]:
t0 = time.time()
anio_ref = sql("""SELECT anio FROM (SELECT anio, COUNT(DISTINCT mes) AS meses FROM plata_sfc_tarjetas
                  WHERE indicador = 'tx_compras_nal' GROUP BY anio) WHERE meses = 12 ORDER BY anio DESC LIMIT 1""").collect()[0][0]
c4 = sql(f"""
WITH kpi AS (
  SELECT producto, AVG(Class) AS tasa_fraude,
         AVG(CASE WHEN Class = 1 THEN Amount END) AS monto_medio_fraude_eur
  FROM plata_transacciones GROUP BY producto),
tx AS (
  SELECT producto, entidad, SUM(valor) AS tx_compras
  FROM plata_sfc_tarjetas
  WHERE anio = {anio_ref} AND indicador IN ('tx_compras_nal', 'tx_compras_ext')
  GROUP BY producto, entidad)
SELECT {anio_ref} AS anio, tx.entidad, tx.tx_compras,
       ROUND(tx.tx_compras * kpi.tasa_fraude)                          AS fraudes_estimados,
       ROUND(tx.tx_compras * kpi.tasa_fraude * kpi.monto_medio_fraude_eur, 0) AS monto_expuesto_eur,
       RANK() OVER (ORDER BY tx.tx_compras DESC)                        AS ranking,
       ROUND(tx.tx_compras / SUM(tx.tx_compras) OVER () * 100, 2)        AS participacion_pct,
       ROUND(SUM(tx.tx_compras) OVER (ORDER BY tx.tx_compras DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
             / SUM(tx.tx_compras) OVER () * 100, 2)                     AS participacion_acum_pct
FROM tx JOIN kpi ON tx.producto = kpi.producto
ORDER BY ranking
""")
guardar(c4, "oro", "exposicion_entidad")
e = c4.toPandas()
mostrar(c4, 15)
registrar("C4_join_exposicion", t0)
top5 = e.head(5)
n80 = int((e.participacion_acum_pct < 80).sum()) + 1
print(f"📝 En {anio_ref}, las 5 entidades con más compras con tarjeta de crédito ({', '.join(top5.entidad.str.title())}) suman {top5.participacion_pct.sum():.1f} % del total; {n80} entidades concentran el 80 %.")
print(f"📝 Con la tasa de referencia de F1, el escenario estima {e.fraudes_estimados.sum():,.0f} transacciones fraudulentas en {anio_ref}: el esfuerzo de detección debe priorizar a las entidades líderes.")

### C5 · ¿Cuánto crece el volumen de compras con tarjeta en Colombia? (P1 · ventana `LAG`)

In [ ]:
t0 = time.time()
c5 = sql("""
WITH mensual AS (
  SELECT fecha, anio, mes, SUM(valor) AS tx_compras_nal
  FROM plata_sfc_tarjetas WHERE indicador = 'tx_compras_nal' GROUP BY fecha, anio, mes)
SELECT fecha, anio, mes, tx_compras_nal,
       LAG(tx_compras_nal, 12) OVER (ORDER BY fecha) AS tx_mismo_mes_anio_anterior,
       ROUND((tx_compras_nal / LAG(tx_compras_nal, 12) OVER (ORDER BY fecha) - 1) * 100, 2) AS crecimiento_anual_pct,
       ROUND(AVG(tx_compras_nal) OVER (ORDER BY fecha ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)) AS media_movil_12m
FROM mensual ORDER BY fecha
""")
guardar(c5, "oro", "tendencia_sfc")
t = c5.toPandas()
mostrar(c5.orderBy(F.desc("fecha")), 13)
registrar("C5_tendencia_sfc", t0)
ult = t.dropna(subset=["crecimiento_anual_pct"]).iloc[-1]
print(f"📝 En {int(ult.mes):02d}/{int(ult.anio)} se registraron {ult.tx_compras_nal:,.0f} compras nacionales con tarjeta de crédito, {ult.crecimiento_anual_pct:+.1f} % frente al mismo mes del año anterior.")
print(f"📝 Entre {t.anio.min()} y {t.anio.max()} el volumen mensual pasó de {t.tx_compras_nal.iloc[0]:,.0f} a {t.tx_compras_nal.iloc[-1]:,.0f}: más volumen implica más fraude potencial y justifica un tratamiento de Big Data.")

### C6 · ¿Por qué una transacción es fraude? (P3)
Diferencia estandarizada de medias entre fraude y normal para cada variable: |d| grande = la variable separa bien las clases.

In [ ]:
t0 = time.time()
variables = [f"V{i}" for i in range(1, 29)] + ["log_amount", "hora"]
exprs = []
for v in variables:
    exprs.append(f"""SELECT '{v}' AS variable,
        AVG(CASE WHEN Class = 1 THEN {v} END) AS media_fraude,
        AVG(CASE WHEN Class = 0 THEN {v} END) AS media_normal,
        STDDEV({v}) AS desv_total FROM plata_transacciones""")
c6 = sql(" UNION ALL ".join(exprs)) \
    .withColumn("diferencia_estandarizada", F.round((F.col("media_fraude") - F.col("media_normal")) / F.col("desv_total"), 3)) \
    .withColumn("abs_d", F.abs("diferencia_estandarizada")).orderBy(F.desc("abs_d"))
guardar(c6, "oro", "diferencias_variables")
d = c6.toPandas()
mostrar(c6, 10)
registrar("C6_variables", t0)
t3 = d.head(3)
print(f"📝 Las variables que más separan el fraude son {', '.join(t3.variable)} (diferencias de {', '.join(f'{x:+.2f}' for x in t3.diferencia_estandarizada)} desviaciones estándar).")
print(f"📝 El monto (log_amount, d = {d[d.variable=='log_amount'].diferencia_estandarizada.iloc[0]:+.2f}) y la hora (d = {d[d.variable=='hora'].diferencia_estandarizada.iloc[0]:+.2f}) casi no separan las clases: el patrón del fraude está en el comportamiento (V), no en cuánto ni cuándo.")

## Resumen de KPI (evidencia A1.1 / A2.2)

In [ ]:
resumen = pd.DataFrame([
    {"KPI": "KPI1 Tasa de fraude", "valor": f"{k.kpi1_tasa_fraude_pct:.4f} %", "formula": "fraudes / transacciones × 100", "fuente": "plata.transacciones"},
    {"KPI": "KPI2 Monto fraudado", "valor": f"{k.kpi2_monto_fraudado_eur:,.2f} EUR ({k.pct_monto_fraudado:.3f} % del total)", "formula": "Σ Amount con Class = 1", "fuente": "plata.transacciones"},
    {"KPI": "KPI3 Fraudes detectados (recall)", "valor": "ver notebook 04a_modelo", "formula": "VP / (VP + FN) × 100", "fuente": "oro.predicciones"},
])
resumen.to_csv(f"{EVID}/tablas/kpis.csv", index=False)
mostrar(resumen)

## Cierre
Respalda el Lakehouse (en Colab) y confirma la ejecución completa.

In [ ]:
respaldar_lake()
print("✅ Notebook", NOTEBOOK, "terminado · run_id:", RUN_ID)